In [15]:
from pytential import sympy_pytential, min_pytential, quad_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go
from plotly.subplots import make_subplots

This notebook explores the difference between implementing the volume / lattice constraint as a penalty method or Lagrange multipliers.

In [2]:
c0, c1, V = symbols('c0, c1, V')
kappa = 100000

RT = 8.134*300
f_sym = c0*RT*(0+log(c0/(c0+c1))) +c1*RT*(0+log(c1/(c0+c1)))
lattice_constraint = c0 + c1 - V
f_lm = sympy_pytential(f_sym, constraints_sym = [lattice_constraint])


In [3]:
f_penalty_sym = f_sym + (c0+c1)*kappa/2*(log(V/(c0+c1)))**2
f_p = sympy_pytential(f_penalty_sym)

In [4]:
print(f_p(c0=.5, c1=.5, V=1))
print(f_lm(c0=.5, c1=.5, V=0))

-1691.4177500023784
-1691.4177500023784


## Find $f_p(c_0,V)$ through reducing away c1 with minimization

In [5]:
f_p_c0V = min_pytential(f_p, free_vars=['c0', 'V'])
f_lm_c0V = min_pytential(f_lm, free_vars=['c0', 'V'])

In [6]:
x_vals = np.linspace(0.01, 0.99, 10)
X, Y = np.meshgrid(x_vals, x_vals)

In [7]:
F_p_c0V = f_p_c0V(c0=X.ravel(), V=Y.ravel()).reshape(X.shape)

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)


In [8]:
F_lm_c0V = f_lm_c0V(c0=X.ravel(), V=Y.ravel()).reshape(X.shape)

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:495: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning: Values in x were outside bounds

In [9]:
fig = go.Figure()

Z_limits = [np.min(F_p_c0V), 1000]
F_p_c0V_surf = go.Surface(x=X, y=Y, z=F_p_c0V, colorscale='Viridis', 
                          cmin=Z_limits[0], cmax=Z_limits[1], name='f_p(c0, V)')
F_lm_c0V_surf = go.Surface(x=X, y=Y, z=F_lm_c0V, colorscale='Viridis', 
                           cmin=Z_limits[0], cmax=Z_limits[1], name='f_lm(c0, V)')
fig.add_trace(F_p_c0V_surf)
fig.add_trace(F_lm_c0V_surf)

fig.update_layout(title='Plot of f_p with V=1',
        scene = dict(
          xaxis_title='c0',
          yaxis_title='V',
          zaxis_title='f',
          zaxis_range=Z_limits,
          camera=dict(eye=dict(x=-1, y=-1, z=2))  # Adjust the camera position
          ),
        autosize=False,
        width=700, height=700,
        margin=dict(l=65, r=50, b=65, t=90))
fig.show()

This shows both the penalty and LM methods recover the same potential.

# Quadratic expansion

Let's now examine what happens to the quadratic expansion. Let's pick the point $c_0=.5$, $V=1$

In [10]:
f_lm_quad = quad_pytential.from_homog_pyt(f_lm, y0={'c0':.5, 'c1':.5, 'V':1})
f_p_quad = quad_pytential.from_homog_pyt(f_p, y0={'c0':.5, 'c1':.5, 'V':1})

In [11]:
H = np.array(f_p_quad.hess())
print(H)
print(np.linalg.eigvals(H))  # Eigenvalues of the Hessian matrix
h = H[:2,:2]
print(h)
print(np.linalg.eigvals(h))

[[ 100000.  -100000.  -100000. ]
 [-100000.   102440.2   97559.8]
 [-100000.    97559.8  102440.2]]
[     0.  300000.    4880.4]
[[ 100000.  -100000. ]
 [-100000.   102440.2]]
[  1212.65705694 201227.54294306]


In [12]:
f_p_quad_c0V = f_p_quad.reduce_uncontrained_by_minimization(vars_to_keep=['c0', 'V'])
f_lm_quad_c0V = f_lm_quad.reduce_by_eliminating_linear_constraints(vars_to_keep=['c0', 'V'])

[1, 0]
['V', 'c0', 'c1']
['c0', 'V']
[1, 0]


In [13]:
F_p_quad_c0V = f_p_quad_c0V(c0=X.ravel(), V=Y.ravel()).reshape(X.shape)
F_lm_quad_c0V = f_lm_quad_c0V(c0=X.ravel(), V=Y.ravel()).reshape(X.shape)

In [14]:
fig2 = go.Figure()

Z_limits = [np.min(F_p_quad_c0V), 1000]
F_p_quad_c0V_surf = go.Surface(x=X, y=Y, z=F_p_quad_c0V, colorscale='Viridis', 
                          cmin=Z_limits[0], cmax=Z_limits[1], name='f_p_quad(c0, V)')
F_lm_quad_c0V_surf = go.Surface(x=X, y=Y, z=F_lm_quad_c0V, colorscale='Viridis', 
                           cmin=Z_limits[0], cmax=Z_limits[1], name='f_lm_quad(c0, V)')
fig2.add_trace(F_p_quad_c0V_surf)
fig2.add_trace(F_lm_quad_c0V_surf)

#

fig2.update_layout(title='Plot of f_p with V=1',
        scene = dict(
          xaxis_title='c0',
          yaxis_title='V',
          zaxis_title='f',
          zaxis_range=Z_limits,
          camera=dict(eye=dict(x=-1, y=-1, z=2))  # Adjust the camera position
          ),
        autosize=False,
        width=700, height=700,
        margin=dict(l=65, r=50, b=65, t=90))
fig2.show()